# Preprocessing: `player_defense`  (Jugador — defense)

Análisis y limpieza individual de esta tabla. Rango: **2018-2025** (temporadas completas).
Salida limpia → `data/processed/player_defense_clean.csv`.

Estructura estándar: carga → vista general → columnas → cobertura temporal →
duplicados → distribuciones → limpieza específica → export.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RAW  = Path("../data/raw/fbref_r")      # ajusta si tu estructura difiere
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

PREFIJO = "laliga"
TABLA   = "player_defense"
SEASON_MIN, SEASON_MAX = 2018, 2025
CLAVE = ["Season_End_Year", "Squad", "Player"]

df = pd.read_csv(RAW / f"{PREFIJO}_{TABLA}.csv")
if "Season_End_Year" in df.columns:
    df = df[df["Season_End_Year"].between(SEASON_MIN, SEASON_MAX)].reset_index(drop=True)
print("shape:", df.shape)

shape: (4561, 26)


## 1. Vista general

In [2]:
display(df.head(5))
display(df.tail(5))

,Season_End_Year,Squad,Comp,Player,Nation,Pos,Age,Born,Mins_Per_90,Tkl_Tackles,TklW_Tackles,Def 3rd_Tackles,Mid 3rd_Tackles,Att 3rd_Tackles,Tkl_Challenges,Att_Challenges,Tkl_percent_Challenges,Lost_Challenges,Blocks_Blocks,Sh_Blocks,Pass_Blocks,Int,Tkl+Int,Clr,Err,Url
0,2018,Alavés,La Liga,Martin Agirregabiria,ESP,DF,21,1996.0,20.3,50.0,33,31.0,17.0,2.0,32.0,50.0,64.0,18.0,13.0,7.0,6.0,41,91.0,66.0,0.0,https://fbref.com/en/players/355c883a/Martin-A...
1,2018,Alavés,La Liga,Alexis,ESP,DF,31,1985.0,24.3,33.0,24,24.0,7.0,2.0,20.0,45.0,44.4,25.0,28.0,14.0,14.0,30,63.0,114.0,3.0,https://fbref.com/en/players/db0cd1ae/Alexis
2,2018,Alavés,La Liga,Burgui,ESP,"MF,FW",23,1993.0,11.5,10.0,5,2.0,4.0,4.0,2.0,6.0,33.3,4.0,5.0,0.0,5.0,7,17.0,1.0,0.0,https://fbref.com/en/players/45e304c8/Burgui
3,2018,Alavés,La Liga,Ermedin Demirović,BIH,FW,19,1998.0,2.0,1.0,1,0.0,1.0,0.0,0.0,1.0,0.0,1.0,5.0,0.0,5.0,0,1.0,2.0,0.0,https://fbref.com/en/players/ed79b7d3/Ermedin-...
4,2018,Alavés,La Liga,Adrián Diéguez,ESP,DF,21,1996.0,4.8,10.0,5,5.0,5.0,0.0,6.0,13.0,46.2,7.0,5.0,0.0,5.0,11,21.0,24.0,1.0,https://fbref.com/en/players/70438d35/Adrian-D...


,Season_End_Year,Squad,Comp,Player,Nation,Pos,Age,Born,Mins_Per_90,Tkl_Tackles,TklW_Tackles,Def 3rd_Tackles,Mid 3rd_Tackles,Att 3rd_Tackles,Tkl_Challenges,Att_Challenges,Tkl_percent_Challenges,Lost_Challenges,Blocks_Blocks,Sh_Blocks,Pass_Blocks,Int,Tkl+Int,Clr,Err,Url
4556,2025,Villarreal,La Liga,Nicolas Pépé,CIV,"FW,MF",29-141,1995.0,4.2,3.0,0,1.0,1.0,1.0,0.0,3.0,0.0,3.0,2.0,1.0,1.0,3,6.0,2.0,0.0,https://fbref.com/en/players/57e3f0c7/Nicolas-...
4557,2025,Villarreal,La Liga,Ayoze Pérez,ESP,FW,31-080,1993.0,4.6,4.0,2,1.0,3.0,0.0,1.0,6.0,16.7,5.0,4.0,0.0,4.0,2,6.0,5.0,0.0,https://fbref.com/en/players/819aa8e7/Ayoze-Perez
4558,2025,Villarreal,La Liga,Yeremi Pino,ESP,MF,21-363,2002.0,3.6,9.0,6,2.0,5.0,2.0,5.0,11.0,45.5,6.0,5.0,1.0,4.0,1,10.0,3.0,0.0,https://fbref.com/en/players/540ec57b/Yeremi-Pino
4559,2025,Villarreal,La Liga,Denis Suárez,ESP,MF,30-285,1994.0,0.9,1.0,1,1.0,0.0,0.0,1.0,2.0,50.0,1.0,1.0,1.0,0.0,0,1.0,1.0,0.0,https://fbref.com/en/players/12c50af5/Denis-Su...
4560,2025,Villarreal,La Liga,Ramón Terrats,ESP,MF,23-365,2000.0,0.8,3.0,3,1.0,2.0,0.0,2.0,3.0,66.7,1.0,2.0,1.0,1.0,1,4.0,3.0,0.0,https://fbref.com/en/players/f36016a5/Ramon-Te...


## 2. Columnas: tipos, nulos, únicos

In [3]:
resumen = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_nulos": df.isna().sum(),
    "pct_nulos": (df.isna().mean() * 100).round(1),
    "n_unicos": df.nunique(),
}).sort_values("pct_nulos", ascending=False)
display(resumen)

# valores unicos de columnas de baja cardinalidad
for c in df.columns:
    nu = df[c].nunique(dropna=True)
    if nu <= 15:
        print(f"{c} ({nu}):", sorted(df[c].dropna().unique().tolist()))

,dtype,n_nulos,pct_nulos,n_unicos
Tkl_percent_Challenges,float64,490,10.7,479
Tkl_Tackles,float64,11,0.2,109
Blocks_Blocks,float64,11,0.2,65
Lost_Challenges,float64,11,0.2,74
Tkl_Challenges,float64,11,0.2,56
Att_Challenges,float64,11,0.2,108
Mid 3rd_Tackles,float64,11,0.2,56
Att 3rd_Tackles,float64,11,0.2,19
Def 3rd_Tackles,float64,11,0.2,66
Err,float64,11,0.2,8


Season_End_Year (8): [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Comp (1): ['La Liga']
Pos (10): ['DF', 'DF,FW', 'DF,MF', 'FW', 'FW,DF', 'FW,MF', 'GK', 'MF', 'MF,DF', 'MF,FW']
Err (8): [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]


## 3. Cobertura temporal
Filas por temporada — detecta años incompletos (los datos se congelaron el 18/09/2025;
algunas tablas se actualizaron por última vez en oct-2024).

In [ ]:
if "Season_End_Year" in df.columns:
    por_temp = df["Season_End_Year"].value_counts().sort_index()
    print(por_temp)
    por_temp.plot(kind="bar", figsize=(8,3), title="filas por temporada")
    plt.tight_layout(); plt.show()

## 4. Duplicados por clave natural

In [ ]:
clave = [c for c in CLAVE if c in df.columns]
dup = df.duplicated(subset=clave).sum()
print(f"duplicados exactos por {clave}: {dup}")
if dup:
    display(df[df.duplicated(subset=clave, keep=False)].sort_values(clave).head(10))

## 5. Distribuciones numéricas

In [ ]:
num = df.select_dtypes(include=[np.number]).drop(columns=["Season_End_Year","Born"], errors="ignore")
if len(num.columns):
    n = len(num.columns)
    ncols = 5
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 2.6*nrows))
    for ax, c in zip(np.ravel(axes), num.columns):
        num[c].dropna().hist(bins=30, ax=ax)
        ax.set_title(c, fontsize=8)
    for ax in np.ravel(axes)[n:]:
        ax.axis("off")
    plt.tight_layout(); plt.show()

## 6. Limpieza específica
Pasos estándar + espacio para decisiones propias de esta tabla (documéntalas aquí).

In [ ]:
clean = df.copy()

# columnas numericas disfrazadas de texto -> a numerico
for c in clean.columns:
    if clean[c].dtype == object and c not in ("Squad","Comp","Player","Nation","Pos","Age","Url",
                                              "Home","Away","Date","Day","Time","Round","Venue",
                                              "Referee","Notes","MatchURL","Team_or_Opponent",
                                              "Competition_Name","Gender","Country"):
        conv = pd.to_numeric(clean[c], errors="coerce")
        no_nulos = clean[c].notna().sum()
        if no_nulos and conv.notna().sum() / no_nulos > 0.7:
            clean[c] = conv

# Edad "28-045" -> anhos enteros
if "Age" in clean.columns:
    clean["Age_years"] = pd.to_numeric(clean["Age"].astype(str).str.split("-").str[0], errors="coerce")

# Nation "es ESP" -> codigo de 3 letras
if "Nation" in clean.columns:
    clean["Nation_code"] = clean["Nation"].astype(str).str.split().str[-1].replace("nan", pd.NA)

# >>> decisiones especificas de esta tabla: anhadelas aqui <<<

print("shape tras limpieza:", clean.shape)
clean.head(3)

## 7. Export a data/processed

In [ ]:
out = PROC / f"{PREFIJO}_{TABLA}_clean.csv"
clean.to_csv(out, index=False)
print("guardado:", out, f"({len(clean)} filas)")